# Emoji Sentiment

Are popular emojis generally associated with positive or negative sentiments?

The file `"emoji-sentiment.csv"` provides data on the sentiment associated with various emojis.

Researchers examined 1.6 million tweets across 13 European languages. Each tweet was labeled by annotators as positive (+1), negative (-1), or neutral (0). About 4% of these tweets included emojis.

Columns include:
- `Occurrences [5...max]`: Number of times the emoji appears in the dataset.
- `Position [0...1]`: Average position of the emoji in tweets, from start (0) to end (1).
- `Neg [0...1]`: Percentage of tweets with the emoji that are 'negative'.
- `Neu [0...1]`: Percentage of tweets with the emoji that are 'neutral'.
- `Pos [0...1]`: Percentage of tweets with the emoji that are 'positive'.



### Project Ideas:

Data Cleaning: 
- Remove unnecessary columns that are not useful for your analysis.

- Rename the remaining columns using `snake_case` (all lowercase letters with underscores between words).

New Variables:
- Add a new column called `sentiment`, where sentiment = (% positive tweets) - (% negative tweets).

- Add a `positive_flag` column that is `True` if `sentiment > 0` (or above a set threshold), otherwise `False`.

Types of questions you can now answer more easily:
- What percentage of emojis in the dataset have a positive sentiment?

- What percentage of the top 20 most popular emojis are positive?

- Which emoji (with more than 500 mentions) is the most positive?

- Which emoji (with more than 500 mentions) is the most negative?

- Where in the tweets are most emojis located (i.e. at the beginning or the end)?

- Is there a difference in the placement of positive versus negative emojis within a tweet?

## 1. Load and Inspect Data

We start by importing required libraries and loading the dataset.  
Then, we inspect column names to understand the structure.

In [ ]:
# Import pandas for data manipulation
import pandas as pd

# Load the emoji sentiment dataset
df = pd.read_csv('emoji-sentiment.csv')

# Display the first few rows to verify data structure
df.head()

,Char,Image [twemoji],Unicode codepoint,Occurrences [5...max],Position [0...1],Neg [0...1],Neut [0...1],Pos [0...1],Sentiment bar (c.i. 95%),Unicode name,Unicode block
0,😂,😂,0x1f602,14622,0.805,0.247,0.285,0.468,NaN,FACE WITH TEARS OF JOY,Emoticons
1,❤,❤,0x2764,8050,0.747,0.044,0.166,0.790,NaN,HEAVY BLACK HEART,Dingbats
2,♥,♥,0x2665,7144,0.754,0.035,0.272,0.693,NaN,BLACK HEART SUIT,Miscellaneous Symbols
3,😍,😍,0x1f60d,6359,0.765,0.052,0.219,0.729,NaN,SMILING FACE WITH HEART-SHAPED EYES,Emoticons
4,😭,😭,0x1f62d,5526,0.803,0.436,0.220,0.343,NaN,LOUDLY CRYING FACE,Emoticons


In [2]:
# Print all column names to understand available fields
for col in df.columns:
    print(col)

Char
Image [twemoji]
Unicode codepoint
Occurrences [5...max]
Position [0...1]
Neg [0...1]
Neut [0...1]
Pos [0...1]
Sentiment bar (c.i. 95%)
Unicode name
Unicode block


## 2. Clean and Rename Columns

We remove unnecessary columns, standardize column names, and rename them for clarity.

In [3]:
# Drop irrelevant columns that are not needed for analysis
df = df.drop(columns=[
    'Unicode codepoint', 'Unicode name', 'Unicode block', 'Sentiment bar (c.i. 95%)'
])

# Convert column names to lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Rename columns for readability and consistency
df.rename(columns={
    'image_[twemoji]': 'emoji',
    'position_[0...1]': 'position',
    'occurrences_[5...max]': 'occurences',
    'neg_[0...1]': 'negative_percent',
    'neut_[0...1]': 'neutral_percent',
    'pos_[0...1]': 'positive_percent',
}, inplace=True)

# Display cleaned dataframe
df.head()

,char,emoji,occurences,position,negative_percent,neutral_percent,positive_percent
0,😂,😂,14622,0.805,0.247,0.285,0.468
1,❤,❤,8050,0.747,0.044,0.166,0.790
2,♥,♥,7144,0.754,0.035,0.272,0.693
3,😍,😍,6359,0.765,0.052,0.219,0.729
4,😭,😭,5526,0.803,0.436,0.220,0.343


## 3. Calculate Sentiment Scores

We create new columns to quantify sentiment and flag positive emojis.

In [4]:
# Compute sentiment score as difference between positive and negative percentages
df['sentiment'] = df['positive_percent'] - df['negative_percent']

# Create a boolean flag for positive sentiment
df['positive_flag'] = df['sentiment'] > 0

# Display updated dataframe
df.head()

,char,emoji,occurences,position,negative_percent,neutral_percent,positive_percent,sentiment,positive_flag
0,😂,😂,14622,0.805,0.247,0.285,0.468,0.221,True
1,❤,❤,8050,0.747,0.044,0.166,0.790,0.746,True
2,♥,♥,7144,0.754,0.035,0.272,0.693,0.658,True
3,😍,😍,6359,0.765,0.052,0.219,0.729,0.677,True
4,😭,😭,5526,0.803,0.436,0.220,0.343,-0.093,False


## 4. Sentiment Distribution

We calculate the percentage of emojis with positive sentiment and analyze the top 20 most frequent emojis.

In [5]:
# Filter emojis with positive sentiment
positive_emojis = df[df['sentiment'] > 0]

# Calculate percentage of positive emojis in the dataset
percentage_positive = (len(positive_emojis) / len(df)) * 100
print(f"Percentage of emojis with positive sentiment: {percentage_positive:.2f}%")



Percentage of emojis with positive sentiment: 82.42%


In [6]:
# Identify top 20 most frequently used emojis
top_20 = df.sort_values(by='occurences', ascending=False).head(20)

# Filter positive emojis among top 20
top_20_positive_emojis = top_20[top_20['sentiment'] > 0]

# Calculate percentage of positive emojis among top 20
percentage_positive_20 = (len(top_20_positive_emojis) / len(top_20)) * 100
print(f"Percentage of top 20 most popular emojis that are positive: {percentage_positive_20:.2f}%")

Percentage of top 20 most popular emojis that are positive: 90.00%


## 5. Identify Most Positive and Negative Emojis

We focus on popular emojis (used more than 500 times) and find the most positive and most negative ones.

In [9]:
# Filter emojis with high occurrence count
popular_emojis = df[df['occurences'] > 500]

# Find the most positive emoji
most_positive_emoji = popular_emojis.sort_values(by='sentiment', ascending=False).head(1)
most_positive_emoji[['emoji', 'occurences', 'sentiment', 'positive_percent', 'negative_percent']]

,emoji,occurences,sentiment,positive_percent,negative_percent
1,❤,8050,0.746,0.79,0.044


In [10]:
# Find the most negative emoji
most_negative_emoji = popular_emojis.sort_values(by='sentiment', ascending=True).head(1)
most_negative_emoji[['emoji', 'occurences', 'sentiment', 'positive_percent', 'negative_percent']]

,emoji,occurences,sentiment,positive_percent,negative_percent
23,😒,1385,-0.374,0.217,0.591


## 6. Emoji Position Analysis

We categorize emoji positions within messages and compare positive vs. negative emoji placement.

In [11]:
# Categorize emoji position into 'beginning' or 'end' based on numeric range
df['position_category'] = pd.cut(
    df['position'],
    bins=[0, 0.5, 1],
    labels=['beginning', 'end']
)

# Count total emojis by position category
emoji_position_counts = df['position_category'].value_counts()
print(emoji_position_counts)


position_category
end          639
beginning    112
Name: count, dtype: int64


In [12]:
# Separate positive and negative emojis for comparison
positive_emojis = df[df['sentiment'] > 0]
negative_emojis = df[df['sentiment'] < 0]

# Count occurrences by position category for each sentiment type
positive_counts = positive_emojis['position_category'].value_counts()
negative_counts = negative_emojis['position_category'].value_counts()

# Combine counts into a comparison DataFrame
placement_comparison = pd.DataFrame({
    'Positive Emojis': positive_counts,
    'Negative Emojis': negative_counts
}).fillna(0)

# Display comparison table
placement_comparison

,Positive Emojis,Negative Emojis
position_category,,
end,536,82
beginning,83,17


## 7. Insights and Conclusion

- Most emojis are used at the end of messages, especially positive ones.  
- Positive sentiment dominates overall emoji usage.  
- The most popular emojis tend to express positivity, reinforcing emotional tone in communication.